In [1]:
! pip install datasets
! pip install -U 'accelerate==0.27.0'
! pip install -U 'transformers==4.41.2'

In [2]:
from datasets import load_dataset
from transformers import BertTokenizer, EncoderDecoderModel, Trainer, TrainingArguments
import torch
from transformers import get_linear_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from accelerate import Accelerator
from torch.utils.data import DataLoader

# Inicializar Accelerator
accelerator = Accelerator(mixed_precision='fp16')


# Carregar o dataset
all_dataset = load_dataset("VanessaSchenkel/translation-en-pt", field="data")

# dataset = all_dataset['train'].shuffle(seed=42).select(range(int(len(all_dataset['train']) * 0.50)))
dataset = all_dataset['train'].shuffle(seed=42).select(range(int(len(all_dataset['train']) * 0.0005)))

# Função para preparar os dados no formato correto
def preprocess_function(examples):
    # Extrair inputs e targets
    inputs = [ex["portuguese"] for ex in examples["translation"]]
    targets = [ex["english"] for ex in examples["translation"]]

    # Tokenizar inputs
    model_inputs = tokenizer(inputs, padding="max_length", truncation=True)

    # Tokenizar targets e process labels
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, padding="max_length", truncation=True)["input_ids"]

    # Ensure labels match the input length and shape
    model_inputs["labels"] = labels

    return model_inputs

# Carregar o tokenizer e o modelo BERT
model_name = 'bert-base-multilingual-cased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = EncoderDecoderModel.from_encoder_decoder_pretrained(model_name, model_name)

# Definir o decoder_start_token_id
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.pad_token_id = tokenizer.pad_token_id

/home/paulo/workspaces/fiap-pos-tech-ia-para-devs/01-aulas-gravadas/02-evolucao-da-ia-genia-cloud-ml-e-llms/07-tech-challenge/venv/lib/python3.11/site-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/home/paulo/workspaces/fiap-pos-tech-ia-para-devs/01-aulas-gravadas/02-evolucao-da-ia-genia-cloud-ml-e-llms/07-tech-challenge/venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertLMHeadModel were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.0.cross

In [3]:
# Tokenizar o dataset
tokenized_datasets = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/130 [00:00<?, ? examples/s]

/home/paulo/workspaces/fiap-pos-tech-ia-para-devs/01-aulas-gravadas/02-evolucao-da-ia-genia-cloud-ml-e-llms/07-tech-challenge/venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:3946: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [4]:
# Criação de DataLoader para treinamento
from torch.utils.data import DataLoader
train_dataloader = DataLoader(tokenized_datasets, batch_size=16, shuffle=True)

In [5]:

# Configurar os parâmetros de treinamento com ajuste adicional
training_args = TrainingArguments(
    output_dir="./drive/MyDrive/FIAP/LLMs/Aula 2/results",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=1000,
    fp16=True,  # Habilitar mixed precision training
    gradient_accumulation_steps=2,
)


# Inicializar o Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets
)



In [6]:
# Treinar o modelo
trainer.train()

/home/paulo/workspaces/fiap-pos-tech-ia-para-devs/01-aulas-gravadas/02-evolucao-da-ia-genia-cloud-ml-e-llms/07-tech-challenge/venv/lib/python3.11/site-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:623: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than tensor.new_tensor(sourceTensor).
  decoder_attention_mask = decoder_input_ids.new_tensor(decoder_input_ids != self.config.pad_token_id)
/home/paulo/workspaces/fiap-pos-tech-ia-para-devs/01-aulas-gravadas/02-evolucao-da-ia-genia-cloud-ml-e-llms/07-tech-challenge/venv/lib/python3.11/site-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:643: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a m

Step,Training Loss


TrainOutput(global_step=8, training_loss=23.168569564819336, metrics={'train_runtime': 135.9145, 'train_samples_per_second': 0.956, 'train_steps_per_second': 0.059, 'total_flos': 78557130915840.0, 'train_loss': 23.168569564819336, 'epoch': 0.9411764705882353})

In [7]:

# Salvar o modelo
trainer.save_model("./drive/MyDrive/FIAP/LLMs/Aula 2/bert-translation-en-pt-v3")
tokenizer.save_pretrained("./drive/MyDrive/FIAP/LLMs/Aula 2/bert-tokenizer-translation-en-pt-v3")

('./drive/MyDrive/FIAP/LLMs/Aula 2/bert-tokenizer-translation-en-pt-v3/tokenizer_config.json',
 './drive/MyDrive/FIAP/LLMs/Aula 2/bert-tokenizer-translation-en-pt-v3/special_tokens_map.json',
 './drive/MyDrive/FIAP/LLMs/Aula 2/bert-tokenizer-translation-en-pt-v3/vocab.txt',
 './drive/MyDrive/FIAP/LLMs/Aula 2/bert-tokenizer-translation-en-pt-v3/added_tokens.json')